In [1]:
# ── Library imports ──────────────────────────────────────────────────────────
from dotenv import load_dotenv
import os
import openai
import pandas as pd
import json
import time
from typing import Dict, List

In [2]:
# ── API Key ───────────────────────────────────────────────────────────────────
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

In [3]:
# ── File paths ────────────────────────────────────────────────────────────────
BASE_DIR = r"F:\workspace1"

GOV_PATH   = os.path.join(BASE_DIR, "trait_data_gov.csv")
COM_PATH   = os.path.join(BASE_DIR, "trait_data_com.csv")
UNIV_PATH  = os.path.join(BASE_DIR, "trait_data_univ.csv")
PHASE0_PATH = os.path.join(BASE_DIR, "formatted_answer_sheet_phase0.csv")

In [4]:
# ── Policy definitions ────────────────────────────────────────────────────────
mission_oriented_policy = {
    "budget_structure": "Consists of large-scale contracts with significant funding allocations.",
    "schedule": "Long-term schedules are assigned with no allowance for mid-course failure.",
    "development_cost": {
        "funding": "Fully funded by the government.",
        "payment_structure": "Government pays primary development costs.",
        "contract_type": "Cost-plus contracts, with the government covering any cost overruns."
    },
    "mass_production": "Production is government-owned, operated, maintained, and utilized.",
    "contract_form": {
        "type": "Traditional government contracts under the Federal Acquisition Regulation (FAR).",
        "development_contracts": "Major development and production contracts are issued as cost-plus contracts."
    },
    "role_distribution": {
        "lead": "Government-led development with total management responsibility.",
        "industry_role": "Private companies act as subcontractors manufacturing parts and systems.",
        "specs": "Government defines all technical specifications and design requirements."
    },
    "ownership_and_operations": {
        "assets": "Developed rockets and spacecraft are owned by the government.",
        "mission_operations": "Conducted under government supervision with limited commercial use."
    },
    "program_management": {
        "approach": "Strict supervision and process-centered management.",
        "collaboration": "Close collaboration between the government and contractors.",
        "review_approval": "Government review and approval required at every stage."
    },
    "design_approach": "Designs are tailored exactly to requirements, reflecting government and agency standards.",
    "development_risk": "Solely borne by the government, including cost overruns and schedule delays.",
    "operational_risk": "Government is responsible for losses and crew casualties in the event of mission failure.",
    "commercial_risk": {
        "market_relation": "Independent of market conditions.",
        "compensation": "Government provides compensation.",
        "market_use": "Launch vehicles are government-exclusive systems, unrelated to the commercial market.",
        "profit": "Contractors receive predetermined profit from the government."
    },
    "intellectual_property": {
        "accumulation": "Technology and know-how accumulated within government and its contractors.",
        "ownership": "Patents and design information are controlled by the government.",
        "transfer": "Only selected technologies are transferred to the private sector."
    },
    "technology_transfer": {
        "method": "Advanced technologies developed by the government are disseminated through spinoff programs.",
        "private_sector_role": "Private entities apply technologies post-development rather than directly participating in R&D."
    }
}

diffusion_oriented_policy = {
    "budget_structure": "Composed of multiple small-scale contracts with limited funding per unit.",
    "schedule": "Short-term schedules are set, and failure to meet key milestones results in contract termination.",
    "development_cost": {
        "funding": "Cost-sharing between government and private sector.",
        "payment_structure": "Government provides fixed funding; contractors must invest their own capital.",
        "contract_type": "Performance-based contracts, payment issued only upon milestone achievement."
    },
    "mass_production": "Ownership, operation, and maintenance lie with private companies. Government purchases services as needed.",
    "contract_form": {
        "type": "Agreements under space law, allowing flexibility.",
        "development_contracts": "Issued as fixed-price contracts to private entities."
    },
    "role_distribution": {
        "lead": "Joint development between government and private companies.",
        "government_role": "Sets high-level goals and funding; provides technical advice and evaluation only.",
        "industry_role": "Private firms lead design and development efforts."
    },
    "ownership_and_operations": {
        "assets": "Rockets and spacecraft are owned by private companies.",
        "mission_operations": "Government uses them as a service; private firms may serve commercial clients as well."
    },
    "program_management": {
        "approach": "Light supervision with performance-based oversight.",
        "government_involvement": "Validates major milestone achievements only.",
        "decision_making": "Detailed design and development decisions are at company discretion."
    },
    "design_approach": "Designs are goal-oriented and flexible, with multiple approaches allowed. Government specifies outcomes, not methods.",
    "development_risk": {
        "burden": "Shared between government and private companies.",
        "failure_consequence": "Government withholds remaining support; remaining losses are borne by the company."
    },
    "operational_risk": {
        "responsibility": "Private sector under commercial launch oversight agencies.",
        "failure_consequence": "Contractors bear full responsibility for re-launch costs in case of mission failure."
    },
    "commercial_risk": {
        "burden": "Fully borne by the private sector.",
        "ownership": "Launch vehicles are privately owned.",
        "market_responsibility": "Private companies are responsible for securing commercial demand.",
        "revenue_opportunity": "Firms may earn additional revenue from launching third-party satellites."
    },
    "intellectual_property": {
        "ownership": "Technology and know-how are accumulated within private firms.",
        "government_access": "Government retains limited access to necessary data only.",
        "control": "Intellectual property rights remain with the private companies."
    },
    "technology_transfer": {
        "development": "Private firms develop technologies to meet government needs.",
        "government_support": "Government may provide test facilities and technical expertise when necessary.",
        "utilization": "Firms retain and commercialize the technologies after development."
    }
}

# Policy reversal directions
# M2D: Mission → Diffusion
# D2M: Diffusion → Mission
REVERSAL_CONFIG = {
    "M2D": {
        "phase0_col_partner": "M_Partner",
        "phase0_col_reason":  "M_Partner_Reason",
        "new_policy": diffusion_oriented_policy,
        "new_policy_name": "Diffusion-oriented",
        "from_policy_name": "Mission-oriented"
    },
    "D2M": {
        "phase0_col_partner": "D_Partner",
        "phase0_col_reason":  "D_Partner_Reason",
        "new_policy": mission_oriented_policy,
        "new_policy_name": "Mission-oriented",
        "from_policy_name": "Diffusion-oriented"
    }
}

In [5]:
# ── Data loading ──────────────────────────────────────────────────────────────
def load_trait_data():
    """Load trait data for all agent types"""
    gov_data  = pd.read_csv(GOV_PATH,  encoding='utf-8-sig')
    com_data  = pd.read_csv(COM_PATH,  encoding='utf-8-sig')
    univ_data = pd.read_csv(UNIV_PATH, encoding='utf-8-sig')
    print(f"Government: {len(gov_data)} | Companies: {len(com_data)} | Universities: {len(univ_data)}")
    return gov_data, com_data, univ_data


def load_phase0():
    df = pd.read_csv(PHASE0_PATH, encoding='utf-8-sig')
    print(f"Phase 0 loaded: {len(df)} rows")
    return df

In [6]:
# ── System message ────────────────────────────────────────────────────────────
def create_system_message(row, data_type):
    """Create system message per agent (identical to Phase 0)"""
    if data_type == "gov":
        return (
            f"You are a decision-maker for government agency ID {row['ID']}, {row['Name']}. "
            f"Your government agency has the following characteristics: "
            f"Category is {row['Category']}, established in {row['Year']}, "
            f"has {row['Employees']} employees, yearly budget is {row['Budget']}, "
            f"organization information is {row['Organization Info']}, "
            f"main mission is {row['Mission']}, "
            f"subordinate centers are {row['Subordinate Centers']}, "
            f"total number of papers published is {row['Papers']}, "
            f"number of space technology-related patents (B64G) is {row['Patents']}, "
            f"and headquarters location is {row['Location']}."
        )
    elif data_type == "com":
        return (
            f"You are a decision-maker for company ID {row['ID']}, {row['Name']}. "
            f"Your company has the following characteristics: "
            f"Main industry is {row['Main Industry']}, sub industry is {row['Sub Industry']}, "
            f"established in {row['Year']}, has {row['Employees']} employees, "
            f"company type is {row['Company Type']}, revenue is {row['Revenue']}, "
            f"operating income is {row['Operating Income']}, listing status is {row['Listed']}, "
            f"enterprise value is {row['Enterprise Value']}, funding level is {row['Funding']}, "
            f"company information is {row['Company Info']}, "
            f"main products and services are {row['Products Services']}, "
            f"core technology is {row['Core Technology']}, "
            f"number of space technology-related patents (B64G) is {row['Patents']}, "
            f"and headquarters location is {row['Location']}."
        )
    elif data_type == "univ":
        return (
            f"You are a decision-maker for university ID {row['ID']}, {row['Name']}. "
            f"Your university has the following characteristics: "
            f"Category (private/public) is {row['Category']}, established in {row['Year']}, "
            f"annual R&D expenditure is {row['R&D Expenditure']}, "
            f"total number of aerospace department professors is {row['Professors']}, "
            f"number of aerospace graduate students is {row['Graduate Students']}, "
            f"representative laboratory is {row['Laboratory']}, "
            f"main research area is {row['Research Area']}, "
            f"core strength of the aerospace department is {row['Core Strength']}, "
            f"total number of papers published is {row['Papers']}, "
            f"number of space technology-related patents (B64G) is {row['Patents']}, "
            f"and university location is {row['Location']}."
        )

In [7]:
# ── Partner info ──────────────────────────────────────────────────────────────
def create_partner_info(gov_data, com_data, univ_data):
    """
    S3 Policy Reversal: no node addition or removal.
    Partner pool is identical to Phase 0.
    """
    info = "The following is information about potential collaboration partners:\n\n"

    info += "=== Government Agencies ===\n"
    for _, row in gov_data.iterrows():
        info += (
            f"ID {row['ID']}: {row['Name']} - Category: {row['Category']}, "
            f"Established: {row['Year']}, Employees: {row['Employees']}, "
            f"Yearly budget: {row['Budget']}, Organization Info: {row['Organization Info']}, "
            f"Mission: {row['Mission']}, Subordinate Centers: {row['Subordinate Centers']}, "
            f"Papers: {row['Papers']}, Patents: {row['Patents']}, Location: {row['Location']}\n"
        )

    info += "\n=== Companies ===\n"
    for _, row in com_data.iterrows():
        info += (
            f"ID {row['ID']}: {row['Name']} - Industry: {row['Main Industry']}/{row['Sub Industry']}, "
            f"Established: {row['Year']}, Employees: {row['Employees']}, "
            f"Type: {row['Company Type']}, Revenue: {row['Revenue']}, "
            f"Operating Income: {row['Operating Income']}, Listed: {row['Listed']}, "
            f"Enterprise Value: {row['Enterprise Value']}, Funding: {row['Funding']}, "
            f"Company Info: {row['Company Info']}, "
            f"Main Product & Service: {row['Products Services']}, "
            f"Core Technology: {row['Core Technology']}, "
            f"Patents: {row['Patents']}, Location: {row['Location']}\n"
        )

    info += "\n=== Universities ===\n"
    for _, row in univ_data.iterrows():
        info += (
            f"ID {row['ID']}: {row['Name']} - Category(Public/Private): {row['Category']}, "
            f"Established: {row['Year']}, R&D Expenditure: {row['R&D Expenditure']}, "
            f"Professors: {row['Professors']}, Graduate Students: {row['Graduate Students']}, "
            f"Main Laboratory: {row['Laboratory']}, Research Area: {row['Research Area']}, "
            f"Core Strength: {row['Core Strength']}, "
            f"Papers: {row['Papers']}, Patents: {row['Patents']}, Location: {row['Location']}\n"
        )

    return info

In [8]:
# ── Memory builder ────────────────────────────────────────────────────────────
def build_memory(agent_id, prev_result_df, partner_col, reason_col):
    """
    Extract prior-round partner IDs and reasons for a given agent.
    Returns a formatted memory string injected into the next phase prompt.
    """
    row = prev_result_df[prev_result_df['ID'] == agent_id]
    if row.empty:
        return "No prior collaboration record available."

    partners = str(row.iloc[0][partner_col]).strip()
    reasons  = str(row.iloc[0][reason_col]).strip()

    memory = (
        f"[Prior Round Memory]\n"
        f"In the previous round, your organization selected the following collaboration partners:\n"
        f"Partner IDs: {partners}\n"
        f"Reasons for selection:\n{reasons}\n"
    )
    return memory

In [9]:
# ── Prompt builder ────────────────────────────────────────────────────────────
def create_user_prompt_reversal(new_policy_dict, new_policy_name, from_policy_name,
                                 partner_info, memory_str, phase):
    """
    Build user prompt for S3 Policy Reversal.
    Phase 1: shock announcement + memory
    Phase 2-3: adaptation/stabilization + memory
    """
    policy_str = json.dumps(new_policy_dict, indent=2, ensure_ascii=False)

    if phase == 1:
        shock_context = (
            f"[Policy Reversal Shock]\n"
            f"The policy environment has shifted from {from_policy_name} to {new_policy_name}. "
            f"All future collaborations will be governed by the new policy conditions described below.\n"
        )
    elif phase == 2:
        shock_context = (
            f"[Adaptation Round]\n"
            f"The policy environment remains {new_policy_name} (reversed from {from_policy_name}). "
            f"Please further adjust your collaboration strategy accordingly.\n"
        )
    else:  # phase == 3
        shock_context = (
            f"[Stabilization Round]\n"
            f"The policy environment remains {new_policy_name} (reversed from {from_policy_name}). "
            f"Please finalize your collaboration strategy under the stabilized policy environment.\n"
        )

    prompt = f"""
The government will be operating research and development and will be forming markets in the following areas:
Launch Services, Launch Site Operators, Launch Support Services and Equipment, Launch Vehicle Manufacturers,
Launch Vehicle Equipment and Software, Manufacturers of Spacecraft Deployment Systems,
Launch Vehicle Test Equipment, Launch Vehicle Design and Manufacturing Support.

{memory_str}

{shock_context}

New policy direction: {policy_str}

The profiles of potential collaboration partners:
{partner_info}

Based on your prior collaboration history, the policy reversal, and the profiles of potential partners,
please decide your collaboration partners by:
1. Partners to MAINTAIN from the previous round (with reasons)
2. Partners to DISCONTINUE from the previous round (with reasons)
3. NEW partners to add (with reasons)

Please respond using the following format:
- Selected Partner ID(s): (all final partners, numbers only, comma-separated)
- Maintained Partner IDs: 
- Discontinued Partner IDs:
- New Partner IDs:
- Reason(s) for Each Decision:
- Overall Considerations for Your Decision:
"""
    return prompt

In [10]:
# ── OpenAI call ───────────────────────────────────────────────────────────────
def ask_openai(system_message, user_message, max_retries=3):
    """GPT response generation with retry logic"""
    for attempt in range(max_retries):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-4.1",
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user",   "content": user_message},
                ],
                temperature=1.0,
                max_tokens=9000
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"API call failed (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return "API_ERROR"

In [11]:
# ── Result parser ─────────────────────────────────────────────────────────────
def parse_partner_ids(response_text):
    """
    Extract comma-separated partner IDs from LLM response.
    Looks for the line starting with '- Selected Partner ID(s):'
    """
    for line in response_text.split('\n'):
        if 'Selected Partner ID' in line:
            # Extract everything after the colon
            parts = line.split(':', 1)
            if len(parts) > 1:
                ids_str = parts[1].strip()
                # Remove any non-numeric, non-comma, non-space characters
                ids_str = ''.join(c for c in ids_str if c.isdigit() or c in ', ')
                return ids_str.strip()
    return ""

In [12]:
# ── Phase runner ──────────────────────────────────────────────────────────────
def run_phase(phase, reversal_key, gov_data, com_data, univ_data,
              prev_result_df, partner_info):
    """
    Run one phase for a given reversal direction (M2D or D2M).

    Parameters
    ----------
    phase          : int  (1, 2, or 3)
    reversal_key   : str  ('M2D' or 'D2M')
    prev_result_df : DataFrame  results from the immediately preceding phase
    """
    cfg = REVERSAL_CONFIG[reversal_key]

    # Column names in prev_result_df
    if phase == 1:
        # Memory comes from Phase 0 columns (M_Partner / D_Partner)
        mem_partner_col = cfg["phase0_col_partner"]
        mem_reason_col  = cfg["phase0_col_reason"]
    else:
        # Memory comes from previous phase output columns
        mem_partner_col = "Partner_IDs"
        mem_reason_col  = "Full_Response"

    results = []
    all_data = []

    for _, row in gov_data.iterrows():
        all_data.append((row, "gov"))
    for _, row in com_data.iterrows():
        all_data.append((row, "com"))
    for _, row in univ_data.iterrows():
        all_data.append((row, "univ"))

    total = len(all_data)
    for i, (row, data_type) in enumerate(all_data):
        agent_id = row['ID']
        print(f"[{reversal_key}] Phase {phase} | Processing: ID {agent_id} ({i+1}/{total})")

        # System message
        system_msg = create_system_message(row, data_type)

        # Memory from previous phase
        memory_str = build_memory(
            agent_id, prev_result_df,
            mem_partner_col, mem_reason_col
        )

        # User prompt
        user_msg = create_user_prompt_reversal(
            new_policy_dict  = cfg["new_policy"],
            new_policy_name  = cfg["new_policy_name"],
            from_policy_name = cfg["from_policy_name"],
            partner_info     = partner_info,
            memory_str       = memory_str,
            phase            = phase
        )

        # API call
        response = ask_openai(system_msg, user_msg)

        # Parse partner IDs
        partner_ids = parse_partner_ids(response)

        results.append({
            "ID":           agent_id,
            "Partner_IDs":  partner_ids,
            "Full_Response": response
        })

        time.sleep(1)

    result_df = pd.DataFrame(results)

    # Save result
    out_path = os.path.join(
        BASE_DIR,
        f"answer_sheet_S3_{reversal_key}_phase{phase}.csv"
    )
    result_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"Saved: {out_path}")

    return result_df

In [ ]:
# ── Execute ───────────────────────────────────────────────────────────────────
main()